# AI Voice Fraud Detection

Trains a Wav2Vec2-based real-vs-AI-voice classifier, combining:
- **ASVspoof 2019** (auto-downloaded from Hugging Face) for bulk real + fake audio
- **Your own voice clips** (uploaded below)
- **Your ElevenLabs clips** (uploaded below)



## 1. Install dependencies

In [5]:
!pip install -q transformers datasets torchaudio librosa soundfile faster-whisper scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 34.4 MB/s eta 0:00:00


## 2. Set up project folders

In [6]:
import os

BASE = "/content/fraud_detector"
DIRS = [
    f"{BASE}/data/real/own_recordings",
    f"{BASE}/data/real/asvspoof_bonafide",
    f"{BASE}/data/fake/elevenlabs",
    f"{BASE}/data/fake/asvspoof_spoof",
    f"{BASE}/checkpoints",
    f"{BASE}/utils",
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

os.chdir(BASE)
print("Working directory:", os.getcwd())


Working directory: /content/fraud_detector


## 3. Project source files



In [7]:
%%writefile preprocess.py
"""
preprocess.py

Audio loading / cleaning utilities shared by training, evaluation, and
live inference. Keeping this centralized matters: any mismatch between
how training audio and live-call audio are preprocessed will quietly
hurt accuracy.
"""

import torch
import torchaudio
import torchaudio.functional as AF


TARGET_SR = 16000  # Wav2Vec2 expects 16kHz mono input


def load_audio(filepath: str, target_sr: int = TARGET_SR) -> torch.Tensor:
    """Load an audio file, downmix to mono, resample to target_sr."""
    waveform, sr = torchaudio.load(filepath)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)

    return waveform.squeeze(0)  # (num_samples,)


def trim_silence(waveform: torch.Tensor, top_db: float = 30.0) -> torch.Tensor:
    """Trim leading/trailing silence. Falls back gracefully on pure silence."""
    import librosa
    import numpy as np

    audio_np = waveform.numpy()
    trimmed, _ = librosa.effects.trim(audio_np, top_db=top_db)
    if len(trimmed) == 0:
        return waveform
    return torch.from_numpy(trimmed)


def normalize_volume(waveform: torch.Tensor) -> torch.Tensor:
    """Peak-normalize to avoid loudness being a spurious signal."""
    peak = waveform.abs().max()
    if peak > 0:
        waveform = waveform / peak
    return waveform


def pad_or_truncate(waveform: torch.Tensor, max_duration: float = 4.0, sr: int = TARGET_SR) -> torch.Tensor:
    """Fix all clips to the same length for batching."""
    max_samples = int(max_duration * sr)
    if waveform.shape[0] > max_samples:
        return waveform[:max_samples]
    pad_amount = max_samples - waveform.shape[0]
    return torch.nn.functional.pad(waveform, (0, pad_amount))


def simulate_phone_channel(waveform: torch.Tensor, sr: int = TARGET_SR) -> torch.Tensor:
    """
    Data augmentation: roughly approximate what real phone-call audio
    sounds like after codec compression (narrowband, some noise).

    This is optional but strongly recommended -- training only on clean
    studio-quality clips (common in public datasets) and then deploying
    on real phone audio is a common cause of accuracy dropping in
    production. Use this as an augmentation during training, applied to
    a fraction of batches, not on every sample.
    """
    # Band-limit to simulate narrowband telephony (~300-3400 Hz)
    waveform = AF.highpass_biquad(waveform, sr, cutoff_freq=300)
    waveform = AF.lowpass_biquad(waveform, sr, cutoff_freq=3400)

    # Light Gaussian noise to simulate line noise
    noise = torch.randn_like(waveform) * 0.005
    waveform = waveform + noise

    return waveform


def preprocess_pipeline(
    filepath: str,
    target_sr: int = TARGET_SR,
    max_duration: float = 4.0,
    trim: bool = True,
    normalize: bool = True,
    augment_phone_channel: bool = False,
) -> torch.Tensor:
    """Full preprocessing chain from filepath to a fixed-length waveform tensor."""
    waveform = load_audio(filepath, target_sr)

    if trim:
        waveform = trim_silence(waveform)

    if normalize:
        waveform = normalize_volume(waveform)

    if augment_phone_channel:
        waveform = simulate_phone_channel(waveform, target_sr)

    waveform = pad_or_truncate(waveform, max_duration, target_sr)

    return waveform


Writing preprocess.py


In [8]:
%%writefile model.py
"""
model.py

Wav2Vec2-based binary classifier for real vs AI-generated (spoofed) voice
detection. Uses a pretrained Wav2Vec2 encoder as a feature extractor and
adds a lightweight classification head on top.

Label convention (used throughout this project):
    0 = genuine / bona fide human voice
    1 = spoof / AI-generated (TTS, voice conversion, replay, etc.)
"""

import torch
import torch.nn as nn
from transformers import Wav2Vec2Model


class Wav2Vec2SpoofClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str = "facebook/wav2vec2-base",
        freeze_feature_extractor: bool = True,
        freeze_encoder_layers: int = 0,
        num_labels: int = 2,
        classifier_hidden_size: int = 256,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(pretrained_model_name)

        # Freeze the CNN feature extractor (raw-waveform front end) - standard
        # practice when fine-tuning wav2vec2 on a downstream task with a
        # modest amount of data.
        if freeze_feature_extractor:
            self.wav2vec2.feature_extractor._freeze_parameters()

        # Optionally freeze the first N transformer encoder layers to reduce
        # overfitting / speed up training when the dataset is small.
        if freeze_encoder_layers > 0:
            for i, layer in enumerate(self.wav2vec2.encoder.layers):
                if i < freeze_encoder_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

        hidden_size = self.wav2vec2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, classifier_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_size, num_labels),
        )

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor = None):
        """
        input_values: (batch, num_samples) raw waveform, already processed
                       by a Wav2Vec2FeatureExtractor / processor.
        attention_mask: optional (batch, num_samples) mask for padded audio.

        Returns:
            logits: (batch, num_labels)
        """
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden)

        # Mean-pool over the time dimension to get a fixed-size utterance
        # embedding. (Attention-weighted pooling is a reasonable upgrade
        # later if you want to squeeze out more accuracy.)
        pooled = hidden_states.mean(dim=1)

        logits = self.classifier(pooled)
        return logits

    def predict_fake_probability(self, input_values: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        """Convenience method: returns P(spoof) for each item in the batch."""
        self.eval()
        with torch.no_grad():
            logits = self.forward(input_values, attention_mask)
            probs = torch.softmax(logits, dim=-1)
            return probs[:, 1]  # probability of class 1 = spoof/fake


Writing model.py


In [9]:
%%writefile dataset.py
"""
dataset.py

PyTorch Dataset that reads a manifest CSV and yields preprocessed audio
tensors ready for the Wav2Vec2 processor.

Manifest CSV format (see utils/manifest_builder.py to generate one):
    filepath,label,source
    data/real/clip001.wav,0,own_recording
    data/fake/elevenlabs/clip002.wav,1,elevenlabs
    data/fake/asvspoof/clip003.wav,1,asvspoof2019

label: 0 = genuine, 1 = spoof
source: free-text tag identifying where the clip came from / which TTS
        engine generated it. Used later to measure per-source accuracy
        and to hold specific sources out of training for a true
        generalization test.
"""

import random
import torch
import pandas as pd
from torch.utils.data import Dataset

from preprocess import preprocess_pipeline, TARGET_SR


class AudioSpoofDataset(Dataset):
    def __init__(
        self,
        manifest_csv: str,
        processor,
        target_sr: int = TARGET_SR,
        max_duration: float = 4.0,
        augment_phone_channel_prob: float = 0.0,
    ):
        self.df = pd.read_csv(manifest_csv)
        self.processor = processor
        self.target_sr = target_sr
        self.max_duration = max_duration
        self.augment_prob = augment_phone_channel_prob

        required_cols = {"filepath", "label"}
        missing = required_cols - set(self.df.columns)
        if missing:
            raise ValueError(f"Manifest is missing required columns: {missing}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        augment = random.random() < self.augment_prob

        waveform = preprocess_pipeline(
            row["filepath"],
            target_sr=self.target_sr,
            max_duration=self.max_duration,
            augment_phone_channel=augment,
        )

        inputs = self.processor(
            waveform.numpy(), sampling_rate=self.target_sr, return_tensors="pt"
        )
        input_values = inputs.input_values.squeeze(0)

        return {
            "input_values": input_values,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "source": row["source"] if "source" in row else "unknown",
        }


def collate_fn(batch):
    """Pads variable-length input_values to the max length in the batch."""
    input_values = [item["input_values"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])
    sources = [item["source"] for item in batch]

    max_len = max(v.shape[0] for v in input_values)
    padded = torch.zeros(len(input_values), max_len)
    for i, v in enumerate(input_values):
        padded[i, : v.shape[0]] = v

    return {"input_values": padded, "label": labels, "source": sources}


Writing dataset.py


In [10]:
%%writefile utils/manifest_builder.py
"""
utils/manifest_builder.py

Walks your data directories and builds a manifest CSV combining:
  - your own genuine recordings
  - public spoof datasets (ASVspoof, WaveFake, In-the-Wild, etc.)
  - your own ElevenLabs (and other TTS engine) generated clips

Expected directory layout (flexible -- pass your own paths via CLI args):

    data/
      real/
        own_recordings/*.wav
        librispeech/*.wav
      fake/
        elevenlabs/*.wav
        asvspoof2019/*.wav
        wavefake/*.wav

Usage:
    python utils/manifest_builder.py \
        --real-dirs data/real/own_recordings data/real/librispeech \
        --fake-dirs data/fake/elevenlabs data/fake/asvspoof2019 data/fake/wavefake \
        --output manifest.csv \
        --holdout-source elevenlabs \
        --holdout-fraction 1.0

The --holdout-source option is important: it lets you keep an entire TTS
engine's clips OUT of the training set (or the specified fraction of them)
so your test set can measure generalization to unseen fake-voice
generators, which is the realistic production failure mode.
"""

import argparse
import csv
import os
import random

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}


def collect_files(directory: str):
    files = []
    for root, _, filenames in os.walk(directory):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in AUDIO_EXTENSIONS:
                files.append(os.path.join(root, f))
    return files


def build_manifest(real_dirs, fake_dirs, output_csv, holdout_source=None,
                    holdout_fraction=1.0, val_split=0.1, test_split=0.1, seed=42):
    random.seed(seed)
    rows = []  # (filepath, label, source, split)

    for d in real_dirs:
        source_name = os.path.basename(os.path.normpath(d))
        for f in collect_files(d):
            rows.append([f, 0, source_name])

    for d in fake_dirs:
        source_name = os.path.basename(os.path.normpath(d))
        for f in collect_files(d):
            rows.append([f, 1, source_name])

    if not rows:
        raise RuntimeError("No audio files found in the provided directories.")

    random.shuffle(rows)

    # Assign splits. If holdout_source is set, force that fraction of that
    # source's clips into the test set only (never train/val), so it acts
    # as a true unseen-generator generalization check.
    holdout_rows, remaining_rows = [], []
    if holdout_source:
        for row in rows:
            if row[2] == holdout_source and random.random() < holdout_fraction:
                holdout_rows.append(row)
            else:
                remaining_rows.append(row)
    else:
        remaining_rows = rows

    n = len(remaining_rows)
    n_val = int(n * val_split)
    n_test = int(n * test_split)

    val_rows = remaining_rows[:n_val]
    test_rows = remaining_rows[n_val:n_val + n_test]
    train_rows = remaining_rows[n_val + n_test:]
    test_rows = test_rows + holdout_rows  # held-out source clips always land in test

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filepath", "label", "source", "split"])
        for split_name, split_rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
            for filepath, label, source in split_rows:
                writer.writerow([filepath, label, source, split_name])

    print(f"Manifest written to {output_csv}")
    print(f"  train: {len(train_rows)}  val: {len(val_rows)}  test: {len(test_rows)}")
    if holdout_source:
        print(f"  (held out {len(holdout_rows)} clips of source '{holdout_source}' into test only)")


def split_manifest_by_column(input_csv: str, split_col: str = "split"):
    """
    Helper: given the combined manifest with a 'split' column, write out
    separate train.csv / val.csv / test.csv (the format dataset.py expects).
    """
    import pandas as pd
    df = pd.read_csv(input_csv)
    base = os.path.splitext(input_csv)[0]
    for split_name in ["train", "val", "test"]:
        subset = df[df[split_col] == split_name].drop(columns=[split_col])
        out_path = f"{base}_{split_name}.csv"
        subset.to_csv(out_path, index=False)
        print(f"Wrote {out_path} ({len(subset)} rows)")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--real-dirs", nargs="+", required=True)
    parser.add_argument("--fake-dirs", nargs="+", required=True)
    parser.add_argument("--output", default="manifest.csv")
    parser.add_argument("--holdout-source", default=None,
                         help="Source name to hold out of train/val entirely (generalization test)")
    parser.add_argument("--holdout-fraction", type=float, default=1.0)
    parser.add_argument("--val-split", type=float, default=0.1)
    parser.add_argument("--test-split", type=float, default=0.1)
    args = parser.parse_args()

    build_manifest(
        args.real_dirs, args.fake_dirs, args.output,
        holdout_source=args.holdout_source,
        holdout_fraction=args.holdout_fraction,
        val_split=args.val_split,
        test_split=args.test_split,
    )
    split_manifest_by_column(args.output)


Writing utils/manifest_builder.py


In [11]:
%%writefile train.py
"""
train.py

Fine-tunes the Wav2Vec2SpoofClassifier on your manifest CSVs.

Usage:
    python train.py \
        --train-manifest manifest_train.csv \
        --val-manifest manifest_val.csv \
        --epochs 10 \
        --batch-size 8 \
        --lr 1e-5 \
        --output-dir checkpoints/
"""

import argparse
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import Wav2Vec2FeatureExtractor, get_linear_schedule_with_warmup
from tqdm import tqdm

from model import Wav2Vec2SpoofClassifier
from dataset import AudioSpoofDataset, collate_fn


def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    total_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in dataloader:
            input_values = batch["input_values"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_values)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    processor = Wav2Vec2FeatureExtractor.from_pretrained(args.pretrained_model)

    train_dataset = AudioSpoofDataset(
        args.train_manifest, processor,
        max_duration=args.max_duration,
        augment_phone_channel_prob=args.augment_prob,
    )
    val_dataset = AudioSpoofDataset(
        args.val_manifest, processor,
        max_duration=args.max_duration,
        augment_phone_channel_prob=0.0,  # no augmentation on validation
    )

    train_loader = DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=args.num_workers,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=args.batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=args.num_workers,
    )

    model = Wav2Vec2SpoofClassifier(
        pretrained_model_name=args.pretrained_model,
        freeze_feature_extractor=True,
        freeze_encoder_layers=args.freeze_layers,
    ).to(device)

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr)
    total_steps = len(train_loader) * args.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    criterion = torch.nn.CrossEntropyLoss()

    os.makedirs(args.output_dir, exist_ok=True)
    best_val_acc = 0.0

    for epoch in range(1, args.epochs + 1):
        model.train()
        epoch_loss = 0.0
        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{args.epochs}")

        for batch in progress:
            input_values = batch["input_values"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_values)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        avg_train_loss = epoch_loss / len(train_loader)
        val_loss, val_acc = evaluate(model, val_loader, device)

        print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        # Save latest checkpoint every epoch
        torch.save(model.state_dict(), os.path.join(args.output_dir, "last.pt"))

        # Save best checkpoint by validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(args.output_dir, "best.pt"))
            print(f"  -> new best model saved (val_acc={val_acc:.4f})")

    print(f"Training complete. Best val accuracy: {best_val_acc:.4f}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-manifest", required=True)
    parser.add_argument("--val-manifest", required=True)
    parser.add_argument("--pretrained-model", default="facebook/wav2vec2-base")
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-5)
    parser.add_argument("--max-duration", type=float, default=4.0)
    parser.add_argument("--freeze-layers", type=int, default=0,
                         help="Number of leading transformer layers to freeze")
    parser.add_argument("--augment-prob", type=float, default=0.3,
                         help="Probability of applying phone-channel simulation per sample")
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--output-dir", default="checkpoints")
    args = parser.parse_args()

    train(args)


Writing train.py


In [12]:
%%writefile evaluate.py
"""
evaluate.py

Evaluates a trained checkpoint on a test manifest, reporting:
  - overall accuracy / precision / recall / F1
  - confusion matrix
  - per-source accuracy (crucial for checking generalization to TTS
    engines/datasets that were held out of training)

Usage:
    python evaluate.py --manifest manifest_test.csv --checkpoint checkpoints/best.pt
"""

import argparse
import torch
from torch.utils.data import DataLoader
from transformers import Wav2Vec2FeatureExtractor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
import pandas as pd

from model import Wav2Vec2SpoofClassifier
from dataset import AudioSpoofDataset, collate_fn


def run_evaluation(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    processor = Wav2Vec2FeatureExtractor.from_pretrained(args.pretrained_model)
    dataset = AudioSpoofDataset(args.manifest, processor, max_duration=args.max_duration)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

    model = Wav2Vec2SpoofClassifier(pretrained_model_name=args.pretrained_model).to(device)
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))
    model.eval()

    all_labels, all_preds, all_sources, all_probs = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            input_values = batch["input_values"].to(device)
            labels = batch["label"]
            sources = batch["source"]

            logits = model(input_values)
            probs = torch.softmax(logits, dim=-1)[:, 1]
            preds = logits.argmax(dim=-1).cpu()

            all_labels.extend(labels.tolist())
            all_preds.extend(preds.tolist())
            all_sources.extend(sources)
            all_probs.extend(probs.cpu().tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    print("=== Overall metrics ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 score:  {f1:.4f}")
    print("Confusion matrix (rows=true, cols=pred), [genuine, spoof]:")
    print(cm)

    print("\n=== Per-source accuracy (generalization check) ===")
    df = pd.DataFrame({
        "source": all_sources,
        "label": all_labels,
        "pred": all_preds,
    })
    for source, group in df.groupby("source"):
        source_acc = (group["label"] == group["pred"]).mean()
        print(f"  {source:25s}  n={len(group):4d}  acc={source_acc:.4f}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--pretrained-model", default="facebook/wav2vec2-base")
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--max-duration", type=float, default=4.0)
    args = parser.parse_args()

    run_evaluation(args)


Writing evaluate.py


In [13]:
%%writefile infer.py
"""
infer.py

Run the trained spoof classifier on a single audio file and print the
fake probability score.

Usage:
    python infer.py --audio call_clip.wav --checkpoint checkpoints/best.pt
"""

import argparse
import torch
from transformers import Wav2Vec2FeatureExtractor

from model import Wav2Vec2SpoofClassifier
from preprocess import preprocess_pipeline, TARGET_SR


def load_model(checkpoint_path, pretrained_model="facebook/wav2vec2-base", device="cpu"):
    model = Wav2Vec2SpoofClassifier(pretrained_model_name=pretrained_model)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    return model


def get_fake_probability(audio_path, model, processor, device="cpu", max_duration=4.0):
    waveform = preprocess_pipeline(audio_path, max_duration=max_duration)
    inputs = processor(waveform.numpy(), sampling_rate=TARGET_SR, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    fake_prob = model.predict_fake_probability(input_values).item()
    return fake_prob


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--audio", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--pretrained-model", default="facebook/wav2vec2-base")
    parser.add_argument("--max-duration", type=float, default=4.0)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = Wav2Vec2FeatureExtractor.from_pretrained(args.pretrained_model)
    model = load_model(args.checkpoint, args.pretrained_model, device)

    prob = get_fake_probability(args.audio, model, processor, device, args.max_duration)
    print(f"Fake probability: {prob:.4f}")
    print("Verdict:", "LIKELY FAKE (AI-generated)" if prob > 0.5 else "LIKELY GENUINE")


Writing infer.py


In [14]:
%%writefile liveness_challenge.py
"""
liveness_challenge.py

Implements the "Verify" layer: when the spoof classifier flags a call as
high-risk, we issue a random spoken-phrase challenge and check whether
the caller actually said it.

Important limitation to design around: a static phrase-repeat challenge
can be defeated by a live voice-cloning pipeline that listens to the
prompt and clones a reply in real time. This module verifies WHAT was
said (via ASR transcription + fuzzy match). It does NOT by itself verify
that the response wasn't itself synthesized -- for that, you'd run the
response audio back through the spoof classifier too (the call_pipeline
module below does exactly that).
"""

import random
import string
from difflib import SequenceMatcher

from faster_whisper import WhisperModel


# A pool of short, easy-to-say, hard-to-predict phrases. Mixing words
# and digits makes the challenge harder to pre-record or guess.
WORD_POOL = [
    "orange", "river", "tiger", "pencil", "cloud", "guitar", "window",
    "basket", "silver", "mountain", "candle", "pepper", "jacket", "engine",
]
DIGIT_POOL = list(string.digits)


def generate_random_phrase(num_words: int = 2, num_digits: int = 3) -> str:
    """
    Generates a challenge like: "tiger pencil 4 8 1"
    Regenerated fresh per call so it can't be pre-recorded in advance.
    """
    words = random.sample(WORD_POOL, k=num_words)
    digits = [random.choice(DIGIT_POOL) for _ in range(num_digits)]
    parts = words + digits
    random.shuffle(parts)
    return " ".join(parts)


class LivenessVerifier:
    def __init__(self, whisper_model_size: str = "base", device: str = "cpu"):
        # faster-whisper for ASR transcription of the caller's response.
        # "base" is a good speed/accuracy tradeoff; use "small"/"medium"
        # for better accuracy if latency budget allows.
        self.asr_model = WhisperModel(whisper_model_size, device=device, compute_type="int8")

    def transcribe(self, audio_path: str) -> str:
        segments, _ = self.asr_model.transcribe(audio_path, language="en")
        return " ".join(seg.text for seg in segments).strip().lower()

    def verify(self, audio_path: str, expected_phrase: str, similarity_threshold: float = 0.75) -> dict:
        """
        Returns a dict with the transcription, similarity score, and
        pass/fail verdict against the expected challenge phrase.
        """
        transcribed = self.transcribe(audio_path)
        expected_norm = expected_phrase.strip().lower()

        similarity = SequenceMatcher(None, transcribed, expected_norm).ratio()
        passed = similarity >= similarity_threshold

        return {
            "expected_phrase": expected_phrase,
            "transcribed_text": transcribed,
            "similarity": similarity,
            "passed": passed,
        }


if __name__ == "__main__":
    # Quick manual test of phrase generation (no audio needed)
    for _ in range(3):
        print(generate_random_phrase())


Writing liveness_challenge.py


In [15]:
%%writefile call_pipeline.py
"""
call_pipeline.py

Ties the three layers together end to end:

    Detect  -> spoof classifier gives a fake-probability score
    Verify  -> if score is high, issue a liveness challenge
    Protect -> allow the call, or block + report it

This module provides the orchestration logic and a simulated CLI demo.
Wiring "capture_call_audio" up to a real telephony source (Twilio Media
Streams, a SIP trunk, an app's mic input, etc.) is the integration point
you'll fill in for your actual deployment -- that part is inherently
specific to whatever telephony stack you're using.

Usage (demo with pre-recorded files standing in for live audio):
    python call_pipeline.py \
        --incoming-audio suspicious_call.wav \
        --response-audio challenge_response.wav \
        --checkpoint checkpoints/best.pt
"""

import argparse
import json
import torch
from transformers import Wav2Vec2FeatureExtractor

from model import Wav2Vec2SpoofClassifier
from infer import load_model, get_fake_probability
from liveness_challenge import LivenessVerifier, generate_random_phrase


class CallFraudPreventionPipeline:
    def __init__(self, checkpoint_path, pretrained_model="facebook/wav2vec2-base",
                 fake_prob_threshold=0.6, whisper_model_size="base", device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = Wav2Vec2FeatureExtractor.from_pretrained(pretrained_model)
        self.model = load_model(checkpoint_path, pretrained_model, self.device)
        self.fake_prob_threshold = fake_prob_threshold
        self.liveness_verifier = LivenessVerifier(whisper_model_size, device="cpu")

    def detect(self, incoming_audio_path: str) -> float:
        """Layer 1: Detect. Returns fake probability in [0, 1]."""
        return get_fake_probability(incoming_audio_path, self.model, self.processor, self.device)

    def issue_challenge(self) -> str:
        """Layer 2 (part A): generate the random phrase to prompt the caller with."""
        return generate_random_phrase()

    def verify(self, response_audio_path: str, expected_phrase: str) -> dict:
        """
        Layer 2 (part B): Verify.
        Checks the caller's spoken response AND re-runs the spoof
        classifier on the response audio itself -- catching the case of
        a live voice-cloning pipeline replying to the challenge.
        """
        phrase_result = self.liveness_verifier.verify(response_audio_path, expected_phrase)
        response_fake_prob = self.detect(response_audio_path)

        phrase_result["response_fake_probability"] = response_fake_prob
        phrase_result["response_flagged_as_synthetic"] = response_fake_prob > self.fake_prob_threshold
        return phrase_result

    def process_call(self, incoming_audio_path: str, response_audio_path: str = None) -> dict:
        """
        Layer 3: Protect. Runs the full pipeline and returns a decision.

        If response_audio_path is None but the call is flagged high-risk,
        the result indicates a challenge is needed and returns the phrase
        to prompt for -- in a real system you'd play this via TTS/audio
        prompt to the caller, collect their response, then call
        process_call again (or call verify() directly) with the response.
        """
        fake_prob = self.detect(incoming_audio_path)
        result = {
            "fake_probability": round(fake_prob, 4),
            "risk_level": "high" if fake_prob > self.fake_prob_threshold else "low",
        }

        if fake_prob <= self.fake_prob_threshold:
            result["decision"] = "ALLOW"
            result["reason"] = "Low fake-voice probability; call proceeds normally."
            return result

        # High risk -> liveness challenge required
        if response_audio_path is None:
            challenge_phrase = self.issue_challenge()
            result["decision"] = "CHALLENGE_REQUIRED"
            result["challenge_phrase"] = challenge_phrase
            result["reason"] = "High fake-voice probability; liveness verification needed before allowing the call."
            return result

        challenge_phrase = self.issue_challenge()  # in production, reuse the phrase you actually prompted with
        verify_result = self.verify(response_audio_path, challenge_phrase)
        result["liveness_check"] = verify_result

        if verify_result["passed"] and not verify_result["response_flagged_as_synthetic"]:
            result["decision"] = "ALLOW"
            result["reason"] = "Liveness challenge passed; caller appears to be a real, present human."
        else:
            result["decision"] = "BLOCK_AND_REPORT"
            result["reason"] = "Liveness challenge failed or response audio also flagged as synthetic."

        return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--incoming-audio", required=True)
    parser.add_argument("--response-audio", default=None,
                         help="Audio of caller responding to the liveness challenge (omit to only test Detect layer)")
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--pretrained-model", default="facebook/wav2vec2-base")
    parser.add_argument("--threshold", type=float, default=0.6)
    args = parser.parse_args()

    pipeline = CallFraudPreventionPipeline(
        checkpoint_path=args.checkpoint,
        pretrained_model=args.pretrained_model,
        fake_prob_threshold=args.threshold,
    )

    result = pipeline.process_call(args.incoming_audio, args.response_audio)
    print(json.dumps(result, indent=2))


Writing call_pipeline.py


In [16]:
with open("utils/__init__.py", "w") as f:
    pass
print("Source files written.")


Source files written.


## 4. Upload your own voice clips

Select your ~20 real voice `.wav`/`.mp3` files.

In [17]:
from google.colab import files
import shutil

print("Upload your OWN VOICE clips (real):")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"data/real/own_recordings/{fname}")
print(f"Saved {len(uploaded)} files to data/real/own_recordings/")


Upload your OWN VOICE clips (real):


Saving WhatsApp Audio 2026-09-08 at 9.32.54 PM (1).mpeg to WhatsApp Audio 2026-09-08 at 9.32.54 PM (1).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.54 PM (2).mpeg to WhatsApp Audio 2026-09-08 at 9.32.54 PM (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.54 PM.mpeg to WhatsApp Audio 2026-09-08 at 9.32.54 PM.mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.55 PM (1).mpeg to WhatsApp Audio 2026-09-08 at 9.32.55 PM (1).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.55 PM (2).mpeg to WhatsApp Audio 2026-09-08 at 9.32.55 PM (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.55 PM.mpeg to WhatsApp Audio 2026-09-08 at 9.32.55 PM.mpeg
Saving WhatsApp Audio 2026-09-08 at 9.33.26 PM (1).mpeg to WhatsApp Audio 2026-09-08 at 9.33.26 PM (1).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.33.26 PM.mp4 to WhatsApp Audio 2026-09-08 at 9.33.26 PM.mp4
Saving WhatsApp Audio 2026-09-08 at 9.33.26 PM.mpeg to WhatsApp Audio 2026-09-08 at 9.33.26 PM.mpeg
Saving WhatsApp Audio 2026-09-08 at 9.33.27 PM (1).mp4 to What

## 5. Upload your ElevenLabs (AI-generated) clips

In [18]:
print("Upload your ELEVENLABS clips (AI-generated):")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"data/fake/elevenlabs/{fname}")
print(f"Saved {len(uploaded)} files to data/fake/elevenlabs/")


Upload your ELEVENLABS clips (AI-generated):


Saving ElevenLabs_2026-09-08T20_03_33_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_e2.mp3 to ElevenLabs_2026-09-08T20_03_33_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_e2.mp3
Saving ElevenLabs_2026-09-08T20_12_41_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-08T20_12_41_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3
Saving ElevenLabs_2026-09-09T05_50_06_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-09T05_50_06_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3
Saving ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2 (1).mp3 to ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2 (1).mp3
Saving ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, C

## 6. Download ASVspoof 2019 for bulk training data

This pulls the ASVspoof 2019 Logical Access dataset from Hugging Face and saves
bona-fide (real) and spoof (fake) clips into your data folders.

**Note:** the full dataset is large (tens of thousands of clips). We cap how many
we save per split below (`MAX_PER_CLASS`) to keep download/training time
reasonable on a free Colab GPU — raise it later once your pipeline works
end to end.

The cell first prints one raw example so you can confirm the field names
(`key`/label conventions can vary between dataset versions) before we
commit to a label mapping — check the printed output matches the mapping
below, and adjust if the dataset card indicates otherwise.

In [19]:
from datasets import load_dataset
import soundfile as sf

MAX_PER_CLASS = 800  # increase later once everything works

print("Loading ASVspoof 2019 LA (eval subset, parquet format)...")
asv = load_dataset("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA", split="test", streaming=True)

# Peek at one example to confirm field names / label meaning
example = next(iter(asv))
print("Example record keys:", list(example.keys()))
print(example)

Loading ASVspoof 2019 LA (eval subset, parquet format)...


README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

Example record keys: ['path', 'audio', 'label', 'notes']
{'path': 'LA_E_2834763.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7db3cb1d9160>, 'label': 1, 'notes': '{"utterance_id": "LA_E_2834763", "speaker_id": "LA_0039", "subset": "eval"}'}


In [20]:
# Based on the printed example above, ASVspoof2019 (this HF mirror) uses:
#   'key' == 1  -> bonafide (real)
#   'key' == 0  -> spoof (fake)
# If the printed example above shows different field names or a string
# label instead, update the logic in this cell accordingly before running.

bonafide_count, spoof_count = 0, 0

for i, record in enumerate(asv):
    if bonafide_count >= MAX_PER_CLASS and spoof_count >= MAX_PER_CLASS:
        break

    audio = record["audio"]
    # label: 0 = bonafide (real), 1 = spoof (fake) -- per this dataset's card
    is_bonafide = (record["label"] == 0)

    if is_bonafide and bonafide_count < MAX_PER_CLASS:
        out_path = f"data/real/asvspoof_bonafide/bonafide_{bonafide_count:05d}.wav"
        sf.write(out_path, audio["array"], audio["sampling_rate"])
        bonafide_count += 1
    elif not is_bonafide and spoof_count < MAX_PER_CLASS:
        out_path = f"data/fake/asvspoof_spoof/spoof_{spoof_count:05d}.wav"
        sf.write(out_path, audio["array"], audio["sampling_rate"])
        spoof_count += 1

    if i % 500 == 0:
        print(f"Scanned {i} records... bonafide={bonafide_count} spoof={spoof_count}")

print(f"Done. Saved {bonafide_count} bonafide and {spoof_count} spoof clips.")

Scanned 0 records... bonafide=0 spoof=1
Scanned 500 records... bonafide=59 spoof=442
Scanned 1000 records... bonafide=119 spoof=800
Scanned 1500 records... bonafide=189 spoof=800
Scanned 2000 records... bonafide=236 spoof=800
Scanned 2500 records... bonafide=292 spoof=800
Scanned 3000 records... bonafide=340 spoof=800
Scanned 3500 records... bonafide=398 spoof=800
Scanned 4000 records... bonafide=450 spoof=800
Scanned 4500 records... bonafide=513 spoof=800
Scanned 5000 records... bonafide=561 spoof=800
Scanned 5500 records... bonafide=612 spoof=800
Scanned 6000 records... bonafide=660 spoof=800
Scanned 6500 records... bonafide=707 spoof=800
Scanned 7000 records... bonafide=760 spoof=800
Done. Saved 800 bonafide and 800 spoof clips.


## 7. Build the training manifest

Combines everything: your own voice, your ElevenLabs clips, and the ASVspoof clips.
Your ElevenLabs clips are held out at a set fraction so the test set measures how
well the model generalizes to *your specific* AI voice, not just ASVspoof's older
synthesis systems.

In [21]:
!python utils/manifest_builder.py \
    --real-dirs data/real/own_recordings data/real/asvspoof_bonafide \
    --fake-dirs data/fake/elevenlabs data/fake/asvspoof_spoof \
    --output manifest.csv \
    --holdout-source elevenlabs \
    --holdout-fraction 0.4 \
    --val-split 0.1 \
    --test-split 0.15


Manifest written to manifest.csv
  train: 1206  val: 160  test: 245
  (held out 5 clips of source 'elevenlabs' into test only)
Wrote manifest_train.csv (1206 rows)
Wrote manifest_val.csv (160 rows)
Wrote manifest_test.csv (245 rows)


## 8. Train

Kept small (3 epochs, batch size 8) to fit a free-tier Colab GPU session.
Increase `--epochs` once you've confirmed everything runs end to end.

In [22]:
!python train.py \
    --train-manifest manifest_train.csv \
    --val-manifest manifest_val.csv \
    --epochs 3 \
    --batch-size 8 \
    --lr 1e-5 \
    --augment-prob 0.3 \
    --output-dir checkpoints/


Using device: cuda
preprocessor_config.json: 100% 159/159 [00:00<00:00, 894kB/s]
config.json: 100% 1.84k/1.84k [00:00<00:00, 4.21MB/s]

pytorch_model.bin: downloading bytes:  12% 47.4M/380M [00:00<00:04, 82.3MB/s, 2.34MB/s  ]
pytorch_model.bin: downloading bytes:  19% 72.4M/380M [00:01<00:03, 98.7MB/s, 5.95MB/s  ]
pytorch_model.bin: downloading bytes:  26% 98.2M/380M [00:01<00:02, 109MB/s, 7.96MB/s  ] 
pytorch_model.bin: downloading bytes:  39% 148M/380M [00:01<00:01, 135MB/s, 12.2MB/s  ]
pytorch_model.bin: reconstructing file:  21% 81.1M/380M [00:01<00:05, 57.5MB/s, 5.16MB/s  ]
pytorch_model.bin: downloading bytes:  44% 167M/380M [00:01<00:01, 122MB/s, 14.6MB/s  ]
pytorch_model.bin: downloading bytes:  65% 248M/380M [00:02<00:00, 156MB/s, 20.6MB/s  ]
pytorch_model.bin: downloading bytes: 100% 255M/255M [00:04<00:00, 59.7MB/s, 22.1MB/s  ]
pytorch_model.bin: reconstructing file: 100% 380M/380M [00:04<00:00, 89.0MB/s, 30.8MB/s  ]
Loading weights: 100% 211/211 [00:00<00:00, 28091.61it/s]


## 9. Evaluate — check per-source accuracy, especially on your held-out ElevenLabs clips

In [23]:
!python evaluate.py --manifest manifest_test.csv --checkpoint checkpoints/best.pt


Loading weights: 100% 211/211 [00:00<00:00, 27682.14it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
=== Overall metrics ===
Accuracy:  0.8694
Precision: 0.9794
Recall:    0.7600
F1 score:  0.8559
Confusion matrix (rows=true, cols=pred), [genuine, spoof]:
[[118   2]
 [ 30  95]]

=== Per-source accuracy (generalization check) ===
  asvspoof_bonafide          n= 120  acc=0.9833
  asvspoof_spoof             n= 118  acc=

## 10. (Optional) Save checkpoint to Google Drive so it survives when this Colab session ends

In [24]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

checkpoint_path = "checkpoints/best.pt"
drive_path = "/content/drive/MyDrive/fraud_detector_best.pt"

if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, drive_path)
    print(f"Saved to Google Drive as {drive_path}")
else:
    print(f"Error: Checkpoint file '{checkpoint_path}' not found.")
    print("Please ensure you've run the training cell (step 8) and that training completed successfully to generate the 'best.pt' file.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive as /content/drive/MyDrive/fraud_detector_best.pt


## 11. Try it on a single clip

In [25]:
print("Upload a test clip to classify:")
test_upload = files.upload()
test_filename = list(test_upload.keys())[0]

!python infer.py --audio "{test_filename}" --checkpoint checkpoints/best.pt


Upload a test clip to classify:


Saving ElevenLabs_2026-09-10T00_59_34_Bella - Professional, Bright, Warm_pre_sp87_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-10T00_59_34_Bella - Professional, Bright, Warm_pre_sp87_s50_sb75_se0_b_m2.mp3
Loading weights: 100% 211/211 [00:00<00:00, 27428.20it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Fake probability: 0.9000
Verdict: LIKELY FAKE (AI-generated)


## 12. Full Detect → Verify → Protect pipeline demo

In [26]:
!python call_pipeline.py \
    --incoming-audio "{test_filename}" \
    --checkpoint checkpoints/best.pt


Loading weights: 100% 211/211 [00:00<00:00, 26610.08it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{
  "fake_probability": 0.9,
  "risk_level": "high",
  "decision": "CHALLENGE_REQUIRED",
  "challenge_phrase": "candle river 7 6 3",
  "reason": "High fake-voice probability; liveness verification needed before allowing the call."
}
